<img src=https://upload.wikimedia.org/wikipedia/commons/6/68/Logo_universidad_icesi.svg width=300>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sebastianb92/nlp-labs/blob/main/Session3/1-Transformers.ipynb)


# Maestría en Inteligencia Artificial  
## Procesamiento de Lenguaje natural
### Sesión 3 - Práctica

---


**Integrantes:**  
- Johan Sebastian Bonilla  
- Edwin Gómez  

# Introducción

# Clasificacion de textos - Analisis de reseñas y/u opiniones

En este notebook se aborda el problema de clasificación de texto aplicado al análisis de reseñas y opiniones en lenguaje natural. El objetivo principal es identificar automáticamente el sentimiento  presente en un conjunto de descripciones textuales utilizando técnicas de representación semántica.

# 1. Configurar entorno

En esta sección se configuran las librerías y dependencias necesarias para el análisis de datos y procesamiento de lenguaje natural. Esto garantiza que el entorno esté listo para cargar, limpiar y analizar las conversaciones del chat político.

In [ ]:
import importlib.metadata
import warnings

warnings.filterwarnings('ignore')

installed_packages = [dist.metadata['Name'].lower() for dist in importlib.metadata.distributions()]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y  > /dev/null 2>&1
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y > /dev/null 2>&1
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2 > /dev/null 2>&1
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1 > /dev/null 2>&1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]'==4.41.2 sentence-transformers > /dev/null 2>&1

# 2. Recopilación de datos

Para el presente análisis se utilizará el conjunto de datos mteb/SpanishSentimentClassification, disponible en Hugging Face, el cual contiene reseñas y opiniones en español relacionadas con servicios de hospedaje. Cada registro del dataset se encuentra etiquetado según su polaridad de sentimiento, clasificándose en categorías positivas o negativas.

Este conjunto de datos permitirá evaluar modelos de representación semántica y clasificación automática en una tarea binaria de análisis de sentimiento aplicada a opiniones reales de usuarios.

In [ ]:
from datasets import load_dataset
import warnings
import os

warnings.filterwarnings("ignore")
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
dataset = load_dataset('sepidmnorozy/Spanish_sentiment', split='train')
dataset

In [ ]:
dataset[1]

In [ ]:
for i in range(5):
    print(dataset[i])

In [ ]:
from collections import Counter
Counter(dataset["label"])

In [ ]:
import numpy as np
np.mean([len(t.split()) for t in dataset["text"]])

In [ ]:
text_lengths = [len(row['text']) for row in dataset]
print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

# 3. Analisis Exploratorio



## Estructura del Dataset

In [ ]:
for i in range(5):
    print(dataset[i])

Revisamos los primero 5 registros del dataset para validar estructura y valores

## Análisis de Distribución

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

# Extraer etiquetas
labels = dataset["label"]

# Contar clases
counter = Counter(labels)

# Mostrar conteo
print("Distribución absoluta:")
print(counter)

# Calcular proporciones
total = sum(counter.values())
print("\nDistribución porcentual:")
for cls, freq in counter.items():
    print(f"Clase {cls}: {freq} ({freq/total:.2%})")

# Graficar
plt.figure()
plt.bar(counter.keys(), counter.values())
plt.xticks(list(counter.keys()))
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.title("Distribución de clases")
plt.show()

Podemos observar que se presenta un desbalanceo de clases con un 82.2% de los comentarios positivos en el dataset. El resto pertenecen a comentarios negativos


## Análisis del Texto

In [ ]:
import numpy as np

text_lengths = [len(row['text'].split()) for row in dataset]

print(f"Texto más corto: {min(text_lengths)}")
print(f"Texto más largo: {max(text_lengths)}")
print(f"Longitud promedio: {sum(text_lengths) / len(text_lengths)}")

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99]:
    print(f"{p}%: {np.percentile(text_lengths, p)}")

El promedio de palabras en cada texto es de 16 palabras lo que sugiere textos moderadamente cortos, tipicos en sitios de reviews

## Vocabulario

In [ ]:
from collections import Counter

all_words = []
for text in dataset["text"]:
    all_words.extend(text.split())

vocab = Counter(all_words)
print("Tamaño del vocabulario:", len(vocab))

El dataset cuenta con 3041 palabras unicas lo que indica un vocabulario moderado y una complejidad lexica relativamente baja

## Análisis de Ruido

In [ ]:
import random
for _ in range(5):
    idx = random.randint(0, len(dataset)-1)
    print(dataset[idx])

Podemos ver aleatoriamente algunos registros para rectificar la clasificacion o valor de la etiqueta vs el texto

# 4. Tokenizador

En esta etapa se realiza la construcción y entrenamiento del tokenizador a partir del corpus de texto disponible. Para ello, se utiliza el tokenizador base de GPT-2 mediante la librería Hugging Face Transformers, el cual permite aprender una representación eficiente del lenguaje a partir de los datos.

El proceso consiste en recorrer el dataset por lotes y entrenar un nuevo tokenizador que genera un vocabulario de hasta 50.000 tokens, utilizando como base el alfabeto de bytes convertido a caracteres Unicode. Esto permite que el modelo pueda representar cualquier carácter del texto y manejar adecuadamente palabras desconocidas mediante sub-tokens.

Como resultado, el texto es transformado en secuencias de tokens numéricos, lo que permite que posteriormente los modelos de aprendizaje automático puedan procesar la información textual de manera estructurada y eficiente.

In [ ]:
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from transformers.models.gpt2.tokenization_gpt2 import bytes_to_unicode

length = len(dataset)

tokenizer = AutoTokenizer.from_pretrained("gpt2")

byte_to_unicode_map = bytes_to_unicode()
unicode_to_byte_map = {v: k for k, v in byte_to_unicode_map.items()}
base_vocab = list(unicode_to_byte_map.keys())

def batch_iterator(batch_size=10):
    for i in tqdm(range(0, length, batch_size)):
        yield dataset[i:i+batch_size]

spanish_sentiment_tokenizer = tokenizer.train_new_from_iterator(
    batch_iterator(),
    vocab_size=50000,
    initial_alphabet=base_vocab
)

In [ ]:
spanish_sentiment_tokenizer.pad_token = '[PAD]'

Revisamos el tokenizador obtenido

In [ ]:
tokens = sorted(spanish_sentiment_tokenizer.vocab.items(), key=lambda x: x[1], reverse=False)
print(f"Vocabulario: {spanish_sentiment_tokenizer.vocab_size} tokens")
print("Primeros 15 tokens:")
print([f"{spanish_sentiment_tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[:15]])
print("15 tokens de en medio:")
print([f"{spanish_sentiment_tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[100:120]])
print("Últimos 15 tokens:")
print([f"{spanish_sentiment_tokenizer.convert_tokens_to_string([t])}" for t, _ in tokens[-15:]])

# 5. Definición del Dataset de PyTorch

Ahora definimos una clase Dataset de PyTorch para manejar los datos del problema de clasificación de sentimientos. Esta clase se encargará de:

- Preparar y procesar los textos de entrada.

- Aplicar el tokenizador para convertir el texto en representaciones numéricas.

- Organizar las etiquetas de sentimiento correspondientes.

- Retornar los datos en formato de tensores listos para el entrenamiento del modelo.

In [ ]:
import torch
import numpy as np
from typing import Dict
from torch.utils.data import Dataset

class SpanishSentimentDataset(Dataset):
    """Dataset para clasificación de sentimiento en español"""

    def __init__(self, tokenizer, dataset, seq_length: int = 128):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.seq_length = seq_length
        # Mapeo de etiquetas a índices
        self.id_2_class_map = {0: 'negative', 1: 'positive'}
        self.class_2_id_map = {'negative': 0, 'positive': 1}
        self.num_classes = 2

    def __getitem__(self, index):
        text = self.dataset[index]['text']
        label = self.dataset[index]['label']

        data = {
        k: torch.tensor(v)
        for k, v in self.tokenizer(
            text,
            max_length=self.seq_length,
            truncation=True,
            padding="max_length"
        ).items()
        }
        data["y"] = torch.tensor(label)
        return data

    def __len__(self):
        return len(self.dataset)

### Instanciamos el Dataset y creamos los DataLoaders

Dividimos el dataset en conjuntos de entrenamiento (80%), validación (10%) y prueba (10%).

In [ ]:
from torch.utils.data import random_split, DataLoader

# Configuración
max_len = 128
spanish_sentiment_dataset = SpanishSentimentDataset(spanish_sentiment_tokenizer, dataset, seq_length=max_len)
assert len(spanish_sentiment_dataset) == len(dataset)
batch_size = 8

# Crear dataset
sentiment_dataset = SpanishSentimentDataset(spanish_sentiment_tokenizer, dataset, seq_length=max_len)
print(f"Total de muestras: {len(sentiment_dataset)}")
print(f"Número de clases: {sentiment_dataset.num_classes}")

# División train/val/test
train_dataset, val_dataset, test_dataset = random_split(
    sentiment_dataset,
    lengths=[0.8, 0.1, 0.1],
    generator=torch.Generator().manual_seed(42)
)

# DataLoaders (num_workers=0 para trabajar con CPU local)
num_workers = 2 if IN_COLAB else 0
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

print(f"\nTrain: {len(train_dataset)} muestras, {len(train_loader)} batches")
print(f"Val: {len(val_dataset)} muestras, {len(val_loader)} batches")
print(f"Test: {len(test_dataset)} muestras, {len(test_loader)} batches")

# 6. Definición del Positional Embedding

En esta sección se definen los embeddings utilizados por el modelo Transformer. Se generan las representaciones vectoriales de los tokens y se incorpora información sobre su posición en la secuencia mediante diferentes tipos de codificación posicional (sinusoidal o aprendible), produciendo las representaciones de entrada que serán utilizadas por el modelo.

In [ ]:
import numpy as np
import torch.nn as nn
from enum import Enum
from typing import Optional


class PosEncodingType(Enum):
    SINUSOID = 1
    LEARNABLE = 2


class SinusoidPE(nn.Module):

    def __init__(self, max_len: int, d_model: int):
        super(SinusoidPE, self).__init__()

        # Definimos un vector columna con las posiciones de la secuencia de entrada (pos)
        pos = torch.arange(max_len).unsqueeze(1)
        # Definimos un vector de fila con las dimensiones del embedding (i)
        i = torch.arange(d_model).unsqueeze(0)

        # Calculamos el denominador segun la formula
        div_term = 1 / torch.pow(10000, (2 * (i // 2)) / torch.tensor(d_model, dtype=torch.float32))
        # Aplicamos el denominador a las posiciones
        angle_rads = pos * div_term

        # Inicializamos la matriz de positional encodings
        pos_encoding = torch.zeros(max_len, d_model)
        # Calculamos los embeddings para los numeros pares con seno: PE(pos, 2i)
        pos_encoding[:, 0::2] = torch.sin(angle_rads[:, 0::2])
        # Calculamos los embdeddings para los numeros inpares con coseno: PE(pos, 2i+1)
        pos_encoding[:, 1::2] = torch.cos(angle_rads[:, 1::2])

        # Registramos la variable como atributo de clase
        self.register_buffer("pos_encoding", pos_encoding.unsqueeze(0), persistent=False)


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pos_encoding[:, :x.size(1), :]


class LearnablePE(nn.Module):

    def __init__(self, vocab_size: int, d_model: int, max_len: int = float('-inf')):
        super(LearnablePE, self).__init__()
        self.max_len = max_len
        self.embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        positions = torch.arange(0, max(x.size(-1), self.max_len))
        pos_emb = self.embedding(positions)
        return x + pos_emb



class TokenAndPosEmbedding(nn.Module):

    def __init__(self, max_len: int, embed_dim: int, vocab_size: int, pos_encoding_type: PosEncodingType = PosEncodingType.SINUSOID):
        super(TokenAndPosEmbedding, self).__init__()
        self.token_emb = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        if pos_encoding_type == PosEncodingType.SINUSOID:
            self.pos_emb = SinusoidPE(max_len, embed_dim)
        else:
            self.pos_emb = LearnablePE(vocab_size, embed_dim)


    def forward(self, x):
        token_emb = self.token_emb(x)
        return self.pos_emb(token_emb)

In [ ]:
emb_dim = 128 if not IN_COLAB else 256
tpe = TokenAndPosEmbedding(max_len, emb_dim, spanish_sentiment_tokenizer.vocab_size)
pos_encoding = tpe.pos_emb.pos_encoding.squeeze(0).numpy()

In [ ]:
text = "hola mundo!"
tokens = spanish_sentiment_tokenizer(text, max_length=max_len, truncation=True, padding='max_length')
x = torch.tensor(tokens['input_ids']).unsqueeze(0)
mask = torch.tensor(tokens['input_ids']).unsqueeze(0)
embedding = tpe(x)
embedding.shape

# 7. Definición del Multi-Head Attention

Este módulo permite que el modelo atienda simultáneamente a diferentes partes de la secuencia de entrada mediante múltiples cabezas de atención.

Este proceso se encarga de:

- Generar las representaciones Query, Key y Value a partir de las entradas.

- Calcular los pesos de atención mediante el producto punto escalado.

- Aplicar máscaras para ignorar tokens de relleno cuando sea necesario.

- Combinar las diferentes cabezas de atención para producir la representación final utilizada por el modelo.

In [ ]:
import math


class MultiHeadAttention(nn.Module):

    def __init__(self, embed_size: int, num_heads: int = 8):
        super(MultiHeadAttention, self).__init__()
        self.embed_size = embed_size
        self.num_heads = num_heads
        assert embed_size & num_heads == 0, 'El tamaño del embedding debería ser divisible por el numero de cabezas'
        self.projection_dim = embed_size // num_heads
        self.query = nn.Linear(emb_dim, emb_dim)
        self.key = nn.Linear(emb_dim, emb_dim)
        self.value = nn.Linear(emb_dim, emb_dim)
        self.comibe_heads = nn.Linear(emb_dim, emb_dim)


    @staticmethod
    def _scaled_dot_product(q, k, v, mask=None):
        """scaled dot product.

        Esta función define el bloque mencionado.
        Aquí se hace la multiplicación de matrices
        entre los Q, K y V para luego calcular el
        score de atención.

        Nótese además que aquí aplicamos una máscara
        de atención. Esto se debe a que como estamos
        rellenando las cadenas cortas con un token que
        en si mismo no trae ningún significado, no queremos
        que la red desperdicie recursos operando sobre este
        token, entonces usamos la máscara para poner los valores
        de atención en numeros muy pequeños para que al
        calcular el score, estos no sobresalgan sobre los demás.
        """
        # d_k para el escalamiento
        d_k = q.size()[-1]

        # multiplicacion Q \cdot K^T
        attn_logits = torch.matmul(q, k.transpose(-2, -1))
        # escalamiento
        attn_logits = attn_logits / math.sqrt(d_k)

        # Se aplica la máscara
        if mask is not None:
            attn_logits = attn_logits.masked_fill(mask.reshape(mask.shape[0], 1, 1, -1) == 0, -9e-15)

        # Se calcula el score de atención.
        attention = torch.softmax(attn_logits, dim=-1)
        # Se obtienen los valores tras el score de atención.
        values = torch.matmul(attention, v)
        return values, attention


    def _separate_heads(self, x, batch_size):
        # Llega: (batch, seq_len, emb_dim)
        x =  x.reshape(batch_size, -1, self.num_heads, self.projection_dim)  # (batch, seq_len, num_heads, emb_dim / num_heads)
        return x.permute(0, 2, 1, 3)  # (batch, num_heads, seq_len, emb_dim / num_heads)


    def forward(self, x, mask=None, return_attention=False):
        """forward

        Este es todo el forward pass del multi-head attention.
        Aquí se coordina el resto de las operaciones, como
        la concatenación de las múltiples cabezas como
        el paso por la capa densa previo a entregar el
        resultado.
        """
        # x: (batch, seq_len, emb_dim)
        batch_size, seq_len, emb_dim = x.size()
        q = self.query(x)
        k = self.key(x)
        v = self.value(x)

        q = self._separate_heads(q, batch_size)
        k = self._separate_heads(k, batch_size)
        v = self._separate_heads(v, batch_size)

        weights, attention = self._scaled_dot_product(q, k, v, mask)
        weights = weights.permute(0, 2, 1, 3) # (batch, seq_len, num_heads, emb_dim / num_heads)
        weights = weights.reshape(batch_size, seq_len, emb_dim)
        output = self.comibe_heads(weights)

        if return_attention:
            return output, attention
        else:
            return output

In [ ]:
mha = MultiHeadAttention(emb_dim)
mha(embedding, mask).shape

# 8. Definición del modulo Transformers

En esta sección se define un bloque Transformer que integra los componentes principales de esta arquitectura. El bloque aplica un mecanismo de multi-head attention para capturar relaciones entre los elementos de la secuencia, seguido de una red feed-forward que transforma las representaciones obtenidas. Además, se utilizan dropout y layer normalization para mejorar la estabilidad del entrenamiento y reducir el sobreajuste, produciendo así representaciones más robustas para el modelo.

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, emb_dim: int, num_heads: int = 8):
        super(TransformerBlock, self).__init__()
        self.mhatt = MultiHeadAttention(emb_dim, num_heads)
        self.mhatt_dropput = nn.Dropout(0.2)
        self.ffn = nn.Sequential(
            nn.Linear(emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, emb_dim)
        )
        self.layer_norm1 = nn.LayerNorm(emb_dim)
        self.layer_norm2 = nn.LayerNorm(emb_dim)


    def forward(self, x, mask=None):
        attn_output = self.mhatt(x, mask)
        attn_output = self.mhatt_dropput(attn_output)
        attn_output = self.layer_norm1(attn_output)
        ffn_out = self.ffn(attn_output)
        return self.layer_norm2(ffn_out)

In [ ]:
tb = TransformerBlock(emb_dim)
tb(embedding, mask).shape

In [ ]:
num_heads = 8
vocab_size = spanish_sentiment_tokenizer.vocab_size

token_embeddings = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
transformer = TransformerBlock(emb_dim, num_heads)
ff = nn.Sequential(
    nn.Flatten(),
    nn.Linear(max_len * emb_dim, spanish_sentiment_dataset.num_classes)
)

In [ ]:
it = iter(train_loader)
batch = next(it)
x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']

embeddings = token_embeddings(x)
assert embeddings.shape == (train_loader.batch_size, max_len, emb_dim)

attention = transformer(embeddings, mask)
attention.shape

In [ ]:
pred = ff(attention)
pred.shape

# 9. Definición del clasificador

En esta sección se construye y entrena el modelo de clasificación de sentimientos basado en un Transformer. El proceso sigue los siguientes pasos:

- Se definen los embeddings de tokens y posiciones para representar el texto de entrada.

- Las representaciones pasan por un bloque Transformer con mecanismo de atención para capturar relaciones entre las palabras.

- La salida se envía a una red neuronal densa que actúa como clasificador para predecir el sentimiento.

- Se implementan los pasos de entrenamiento, validación y prueba, junto con el cálculo de métricas de desempeño.

- Finalmente, se configura el optimizador y el Trainer de PyTorch Lightning para entrenar el modelo.

Como se evidenció anteriormente, el dataset presenta un desbalance de clases. Para mitigar este problema, se implementará el uso de class weights, lo que permite penalizar en mayor medida los errores cometidos sobre la clase minoritaria durante el entrenamiento del modelo.

In [ ]:
from collections import Counter

labels = [x['label'] for x in spanish_sentiment_dataset.dataset]

counts = Counter(labels)

class_counts = [counts[i] for i in range(spanish_sentiment_dataset.num_classes)]

print("Distribución de clases:", class_counts)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_lightning import LightningModule, Trainer
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from torchmetrics import Accuracy


class SentimentClassifier(LightningModule):

    def __init__(self, max_len, vocab_size, num_classes, emb_dim, class_counts, num_heads=8):
        super(SentimentClassifier, self).__init__()
        self.num_classes = num_classes

        counts = torch.tensor(class_counts, dtype=torch.float)
        total = counts.sum()
        weights = total / (num_classes * counts)
        self.register_buffer("class_weights", weights)

        self.token_embeddings = TokenAndPosEmbedding(max_len, emb_dim, vocab_size)
        self.transformer = TransformerBlock(emb_dim, num_heads)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(max_len * emb_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

        self.train_acc = Accuracy(task='binary', num_classes=num_classes)
        self.val_acc = Accuracy(task='binary', num_classes=num_classes)
        self.test_acc = Accuracy(task='binary', num_classes=num_classes)



    def forward(self, x, mask=None):
        embeddings = self.token_embeddings(x)
        attention = self.transformer(embeddings, mask)
        return self.classifier(attention)


    def training_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y, weight=self.class_weights)
        preds = torch.argmax(y_hat, dim=1)
        self.train_acc(preds, y)
        self.log('train-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('train-acc', self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        loss = F.cross_entropy(y_hat, y, weight=self.class_weights)
        preds = torch.argmax(y_hat, dim=1)
        self.val_acc(preds, y)
        self.log('val-loss', loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log('val-acc', self.val_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def test_step(self, batch):
        x, mask, y = batch['input_ids'], batch['attention_mask'], batch['y']
        y_hat = self(x, mask)
        preds = torch.argmax(y_hat, dim=1)
        self.test_acc(preds, y)
        self.log('test-acc', self.test_acc, prog_bar=True, on_step=False, on_epoch=True)


    def predict_step(self, batch):
        x, mask = batch['input_ids'], batch['attention_mask']
        return self(x, mask)


    def configure_optimizers(self):
        optimizer =  torch.optim.AdamW(self.parameters(), lr=2e-5, weight_decay=1e-5)
        return optimizer


model = SentimentClassifier(
    max_len=spanish_sentiment_dataset.seq_length,
    vocab_size=spanish_sentiment_tokenizer.vocab_size,
    num_classes=spanish_sentiment_dataset.num_classes,
    emb_dim=emb_dim,
    class_counts=class_counts
)

tb_logger = TensorBoardLogger('tb_logs', name='TransformersClassifier')
callbacks=[EarlyStopping(monitor='train-loss', patience=3, mode='min')]
trainer = Trainer(max_epochs=20, devices=1, logger=tb_logger, callbacks=callbacks, precision="16-mixed")

trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir tb_logs/

In [ ]:
model.eval()
trainer.test(model, test_loader)

El modelo de clasificación de sentimientos basado en Transformer obtuvo una precisión del 85.29% en el conjunto de prueba, lo que significa que el modelo logra clasificar correctamente la mayoría de los comentarios como positivos o negativos. Este resultado muestra que el modelo aprendió a identificar patrones y relaciones entre las palabras de los textos para determinar el tipo de sentimiento expresado en cada comentario.

Aunque el desempeño es bueno, todavía existe un pequeño porcentaje de errores, por lo que el modelo podría mejorarse utilizando más datos de entrenamiento, ajustando algunos parámetros o probando versiones más avanzadas del modelo.

# 10. Hacemos predicciones

In [ ]:
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
predictions = torch.argmax(predictions, dim=-1)
predictions = [spanish_sentiment_dataset.id_2_class_map[pred] for pred in predictions.numpy()]

In [ ]:
import pandas as pd

# Obtener predicciones
predictions = trainer.predict(model, test_loader)
predictions = torch.cat(predictions, dim=0)
pred_labels = torch.argmax(predictions, dim=-1).numpy()

# Obtener índices del test set
test_indices = test_dataset.indices

# Crear DataFrame con resultados
id_2_token = {v: k for k, v in vocab.items()}

df_results = pd.DataFrame({
    'texto': [dataset[i]['text'] for i in test_indices],
    'etiqueta_real': [dataset[i]['label'] for i in test_indices],
    'prediccion': pred_labels,
    'sentimiento_real': [sentiment_dataset.id_2_class_map[dataset[i]['label']] for i in test_indices],
    'sentimiento_pred': [sentiment_dataset.id_2_class_map[p] for p in pred_labels]
})

df_results['correcto'] = df_results['etiqueta_real'] == df_results['prediccion']
print(f"Precisión: {df_results['correcto'].mean():.4f}")
print(f"Total correctos: {df_results['correcto'].sum()} / {len(df_results)}")
df_results.head(10)

In [ ]:
# Análisis de errores
errors = df_results[~df_results['correcto']]
print(f"\nTotal de errores: {len(errors)}")
print(f"\nEjemplos de predicciones incorrectas:")
errors[['texto', 'sentimiento_real', 'sentimiento_pred']].head(10)

# 11. Analizamos los resultados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Matriz de confusión
cm = confusion_matrix(df_results['etiqueta_real'], df_results['prediccion'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Matriz de confusión
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        axes[0].text(j + 0.5, i + 0.5, str(cm[i, j]),
                     ha='center', va='center', color='black', fontsize=14, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'], ax=axes[0])

axes[0].set_xlabel('Predicción')
axes[0].set_ylabel('Real')
axes[0].set_title('Matriz de Confusión')

# Plot 2: Distribución de predicciones
pred_counts = df_results['sentimiento_pred'].value_counts()
axes[1].bar(pred_counts.index, pred_counts.values, color=['salmon', 'lightgreen'])
axes[1].set_xlabel('Sentimiento')
axes[1].set_ylabel('Cantidad')
axes[1].set_title('Distribución de Predicciones')

plt.tight_layout()
plt.show()

# Reporte de clasificación
print("\nReporte de Clasificación:")
print(classification_report(df_results['etiqueta_real'], df_results['prediccion'],
                          target_names=['Negative', 'Positive']))

# 12. Conclusiones

En este notebook se implemento y evaluo un modelo basado en arquitectura Transformer para la clasificación de sentimientos en español, para comparar el desempeño con un modelo LSTM bidireccional.

### Arquitectura

#### Modelo LSTM

- Word embeddings para representar las palabras.

- LSTM bidireccional para capturar dependencias en ambas direcciones del texto.

- Capas densas finales para realizar la clasificación del sentimiento.

#### Modelo Transformer

- Tokenización de los textos para convertirlos en representaciones numéricas.

- Embeddings de tokens y posiciones para representar el contenido y el orden de las palabras.

- Bloque de Multi-Head Attention para capturar relaciones entre palabras dentro de la secuencia.

- Capas densas finales para predecir el sentimiento.

### Resultados

- El modelo LSTM alcanzó una precisión aproximada del 87%, mostrando un buen desempeño general en la clasificación.

- El análisis del reporte de clasificación muestra que el modelo es mucho más preciso identificando comentarios positivos que negativos, lo cual está relacionado con el desbalance de clases presente en el dataset.

- El modelo Transformer obtuvo una precisión cercana al 83% haciendo tratamiento en el balance de clases en el conjunto de prueba, demostrando que también es capaz de aprender patrones relevantes en los textos mediante el mecanismo de atención.

### Comparación entre modelos

- El LSTM logró un desempeño ligeramente superior en precisión general, posiblemente debido a que el dataset es relativamente pequeño.

- El Transformer tiene la ventaja de capturar relaciones globales entre palabras mediante el mecanismo de atención, lo que lo hace especialmente útil en textos más largos o datasets más grandes.

- Ambos modelos muestran buen desempeño en la clase positiva, pero presentan mayores dificultades para clasificar correctamente comentarios negativos.

### Posibles mejoras

- Aumentar el tamaño del dataset para mejorar la capacidad de generalización del modelo.

- Usar modelos preentrenados como BERT o BETO, que suelen ofrecer mejores resultados en tareas de NLP.

- Ajustar hiperparámetros como tamaño de embeddings, número de capas o número de cabezas de atención.

También se evidenció que el uso de modelos basados en arquitecturas de tipo Transformer resulta más provechoso cuando se dispone de datasets de mayor tamaño. Esto se debe a que estos modelos cuentan con un número considerable de parámetros y están diseñados para capturar relaciones complejas y dependencias contextuales dentro del texto, lo cual requiere una cantidad suficiente de datos para ser aprendido adecuadamente.

En conclusión, ambos enfoques muestran buenos resultados para la clasificación de sentimientos en español. Sin embargo, los modelos basados en Transformers ofrecen mayor flexibilidad y escalabilidad, especialmente cuando se trabaja con grandes volúmenes de datos o modelos preentrenados.